# S2. Solving a problem, in Python

The companion of VAMOS-Studio's tutorial *S2. Solving a problem*: the same runs with VAMOS's Python API. Each run is fixed by its seed, so it finds the same front and the same values as the tutorial (measured with VAMOS 1.4.0). It pairs with VAMOS's tutorial [T01. Your first result](https://vamos-optimization.org/tutorials/t01-first-result/).

## Setup

The reference fronts are jMetal's, which VAMOS-Studio ships, and the indicators are computed as the studio computes them: with the objectives normalized so that the reference front spans [0, 1].

In [ ]:
from importlib.resources import files

import numpy as np
import plotly.graph_objects as go
import vamos
from vamos.algorithms import NSGAIIConfig
from vamos.foundation.quality_indicators import get_indicator

print("VAMOS", vamos.__version__)


def reference_front(name: str) -> np.ndarray:
    """A reference front copied from jMetal, as VAMOS-Studio ships it (e.g. "ZDT1.csv")."""
    path = files("vamos_studio") / "resources" / "referenceFronts" / name
    return np.loadtxt(path, delimiter=",", ndmin=2)


def indicators(front: np.ndarray, reference: np.ndarray) -> dict[str, float]:
    """HV and IGD+ as VAMOS-Studio computes them: objectives normalized by the reference front."""
    lower = reference.min(axis=0)
    span = reference.max(axis=0) - lower
    span[span == 0] = 1.0
    normalized, normalized_reference = (front - lower) / span, (reference - lower) / span
    hv = get_indicator("hv", reference_point=np.ones(front.shape[1]))
    igd_plus = get_indicator("igd_plus", reference_front=normalized_reference)
    return {
        "HV": round(float(hv.compute(np.clip(normalized, 0.0, 1.0)).value), 4),
        "IGD+": round(float(igd_plus.compute(normalized).value), 4),
    }


def plot(front: np.ndarray, reference: np.ndarray, title: str) -> go.Figure:
    """A two-objective front in blue over its reference front in grey."""
    figure = go.Figure()
    figure.add_scatter(
        x=reference[:, 0],
        y=reference[:, 1],
        mode="markers",
        name="Reference front",
        marker={"size": 3, "color": "#b7bcc5"},
    )
    figure.add_scatter(
        x=front[:, 0],
        y=front[:, 1],
        mode="markers",
        name="Front",
        marker={"size": 7, "color": "#1769aa"},
    )
    figure.update_layout(title=title, xaxis_title="f1", yaxis_title="f2")
    return figure

## NSGA-II with its default configuration on ZDT1

`vamos.optimize` takes a problem, an algorithm, its configuration and a budget. `NSGAIIConfig.default` is NSGA-II's default configuration, the one VAMOS-Studio starts from, with a population of 100. (Without a configuration, `vamos.optimize` chooses a quick one for a first look, with a population of max(50, 10 × variables) up to 200: 200 on ZDT1, so it would find another front.)

In [ ]:
default = NSGAIIConfig.default(pop_size=100, n_var=30)
result = vamos.optimize(
    "zdt1", algorithm="nsgaii", algorithm_config=default, max_evaluations=25000, seed=1
)
front = np.asarray(result.F)
zdt1 = reference_front("ZDT1.csv")
print(len(front), "solutions", indicators(front, zdt1))  # 100 solutions, HV 0.6593, IGD+ 0.0038
plot(front, zdt1, "NSGA-II on ZDT1").show()

## Another configuration

Start from the default configuration and change two things: a population of 50 and BLX-α crossover (with probability 0.9 and α = 0.5, the values VAMOS-Studio uses).

In [ ]:
import dataclasses

other = dataclasses.replace(
    NSGAIIConfig.default(pop_size=50, n_var=30),
    crossover=("blx_alpha", {"prob": 0.9, "alpha": 0.5}),
)
result = vamos.optimize(
    "zdt1", algorithm="nsgaii", algorithm_config=other, max_evaluations=25000, seed=1
)
front = np.asarray(result.F)
print(len(front), "solutions", indicators(front, zdt1))  # 50 solutions, HV 0.6495, IGD+ 0.0091

## A harder problem: ZDT4

ZDT4 has the same front as ZDT1 but many local fronts. With 10,000 evaluations NSGA-II is still on one of them; with 25,000 it reaches the true front.

In [ ]:
zdt4 = reference_front("ZDT4.csv")
for budget in (10000, 25000):
    result = vamos.optimize(
        "zdt4",
        algorithm="nsgaii",
        algorithm_config=NSGAIIConfig.default(pop_size=100, n_var=10),
        max_evaluations=budget,
        seed=1,
    )
    print(budget, "evaluations:", indicators(np.asarray(result.F), zdt4))
# 10000 evaluations: HV 0.0, IGD+ 0.827; 25000 evaluations: HV 0.6566, IGD+ 0.0054

## Next

Change the problem, the algorithm (`moead`, `smsemoa`, `nsgaiii`…), the budget or the seed, and compare. `vamos.available_problem_names()` lists the problems and `vamos.algorithms.available_algorithms()` the algorithms.